# Notebook 05: Tower damage, outage and losses (Workstream 1D, part 2)
**What this notebook does**
1. Combines the STORM hazard (Notebook 03) with the Fani-calibrated fragility (Notebook 02) for every storm and every tower
2. Turns tower damage into line failures (a line is out if any of its towers is damaged) and, by Monte Carlo, into **population disconnected** from the grid (Raj et al.: substations outside the largest connected network)
3. Prices each storm: **repair cost** (damaged towers × cost per tower) + **outage cost** (energy not supplied × value of lost load)
4. Reports **expected annual damage (EAD)**, the **25/50/100-year losses**, and the **loss-exceedance curve**
5. Ranks lines by risk (input to option (b) in Notebook 06)
6. Validates by re-running historical cyclones (Fani, Phailin, 1999 super cyclone, Yaas …) against recorded tower damage

**Needs:** Notebooks 00, 02, 03 (01 optional). Runtime: about 5–10 minutes.

In [ ]:
# ==== Shared functions (cdri_lib): identical copy in every notebook from 01 onwards ====
# Change a function here, rebuild the notebooks, and every notebook uses the same version.
import os, sys, re, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 160)

KT2MS, NMI2KM, R_EARTH_KM = 0.514444, 1.852, 6371.0
SURF_TO_GRADIENT = 0.85          # surface-to-gradient wind ratio used only to estimate the Holland B parameter
CRS_GEO, CRS_M = 'EPSG:4326', 'EPSG:32645'

# WMO/TD-1555 (Harper, Kepert & Ginger 2010) Table 1.1: 3-s gust factor G(3, To) at +10 m in tropical cyclones
GUST3 = {'in-land':  {60: 1.49, 180: 1.58, 600: 1.66},
         'off-land': {60: 1.36, 180: 1.44, 600: 1.52},
         'off-sea':  {60: 1.23, 180: 1.31, 600: 1.38},
         'at-sea':   {60: 1.11, 180: 1.17, 600: 1.23}}


def setup_project():
    """Mount Google Drive (in Colab) and return the project root and folder dictionary."""
    if 'google.colab' in sys.modules:
        from google.colab import drive
        try:
            drive.mount('/content/drive')
            root = Path('/content/drive/MyDrive/CDRI_Odisha')
        except Exception as e:
            print('Drive mount failed:', e, '\nFix: Runtime → Disconnect and delete runtime, rerun, use the same Google '
                  'account and tick all permission boxes. Using temporary storage for now.')
            root = Path('/content/CDRI_Odisha')
    else:
        root = Path(os.environ.get('CDRI_ROOT', Path.cwd() / 'CDRI_Odisha'))
    dirs = {k: root / v for k, v in {'raw': 'data/raw', 'external': 'data/external', 'interim': 'data/interim',
                                     'processed': 'data/processed', 'figures': 'outputs/figures',
                                     'tables': 'outputs/tables'}.items()}
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    print('Project root:', root)
    return root, dirs


def union(gs):
    """Dissolve a GeoSeries/GeoDataFrame to one geometry (old and new geopandas)."""
    return gs.union_all() if hasattr(gs, 'union_all') else gs.unary_union


def need(path, notebook):
    """Stop with a clear message if an input from an earlier notebook is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f'{path} not found. Run Notebook {notebook} first.')
    return Path(path)


def gust_factor(exposure, avg_s):
    """WMO 3-s gust factor for each exposure class, given the averaging period (s) of the mean wind."""
    return np.array([GUST3[e][int(avg_s)] for e in np.asarray(exposure, dtype=object)], dtype=float)


def haversine_km(lat1, lon1, lat2, lon2):
    la1, lo1, la2, lo2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * R_EARTH_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def _interp_valid(tn, t, y):
    y = np.asarray(y, float)
    ok = np.isfinite(y)
    if ok.sum() == 0:
        return np.full(len(tn), np.nan)
    if ok.sum() == 1:
        return np.full(len(tn), y[ok][0])
    return np.interp(tn, t[ok], y[ok])


def interp_track(t_h, lat, lon, vmax, rmw, pc=None, dt_h=1.0):
    """Interpolate a track to dt_h-hour steps and add the translation velocity (u east, v north; m/s)."""
    t_h = np.asarray(t_h, float)
    if len(t_h) < 2:
        return None
    tn = np.arange(t_h[0], t_h[-1] + 1e-9, dt_h)
    tr = {'t_h': tn, 'lat': _interp_valid(tn, t_h, lat), 'lon': _interp_valid(tn, t_h, lon),
          'vmax': _interp_valid(tn, t_h, vmax), 'rmw': _interp_valid(tn, t_h, rmw),
          'pc': _interp_valid(tn, t_h, pc) if pc is not None else np.full(len(tn), np.nan)}
    if len(tn) < 2:
        tr['u'] = tr['v'] = np.zeros(len(tn))
        return tr
    dy = np.gradient(tr['lat']) * 111.195
    dx = np.gradient(tr['lon']) * 111.195 * np.cos(np.radians(tr['lat']))
    dt = np.gradient(tn) * 3600.0
    tr['u'], tr['v'] = dx * 1000 / dt, dy * 1000 / dt
    return tr


def holland_max_wind(tr, plat, plon, p_env=1008.0, b_fixed=None, b_default=1.6, rho=1.15,
                     max_dist_km=600.0, chunk=4000):
    """Storm-maximum surface wind at points (same averaging basis as tr['vmax']).

    Holland (1980) radial profile scaled to the symmetric maximum wind, plus a translation asymmetry
    (Jelesnianski 1966: Vt·Rm·r/(Rm²+r²), half the translation speed at the RMW, strongest right of track).
    B comes from the central-pressure deficit (Holland 1980) when pressure is known, else b_default; clipped to 1.0–2.5.
    No land-roughness reduction is applied (as in STORM / Bloemendaal 2020); exposure is handled by the gust factor.
    """
    plat, plon = np.asarray(plat, float), np.asarray(plon, float)
    out = np.zeros(len(plat))
    lat, lon = tr['lat'][:, None], tr['lon'][:, None]
    u, v = tr['u'][:, None], tr['v'][:, None]
    vm, rm = tr['vmax'][:, None], np.maximum(tr['rmw'][:, None], 5.0)
    vt = np.hypot(u, v)
    vsym = np.maximum(vm - 0.5 * vt, 0.5 * vm)
    if b_fixed is not None:
        B = np.full_like(vm, float(b_fixed))
    else:
        dp = (p_env - tr['pc'][:, None]) * 100.0
        vg = vsym / SURF_TO_GRADIENT
        B = np.where(np.isfinite(dp) & (dp > 200.0), rho * np.e * vg ** 2 / np.maximum(dp, 200.0), b_default)
        B = np.clip(B, 1.0, 2.5)
    coslat = np.cos(np.radians(lat))
    for s in range(0, len(plat), chunk):
        pl, po = plat[None, s:s + chunk], plon[None, s:s + chunk]
        dy = (pl - lat) * 111.195
        dx = (po - lon) * 111.195 * coslat
        r = np.maximum(np.hypot(dx, dy), 0.5)
        x = (rm / r) ** B
        vs = vsym * np.sqrt(x * np.exp(1.0 - x))
        proj = (-dy * u + dx * v) / r                     # translation component along the cyclonic tangential direction
        w = vs + proj * (rm * r / (rm ** 2 + r ** 2))
        w = np.where(r <= max_dist_km, w, 0.0)
        out[s:s + chunk] = np.maximum(w.max(axis=0), 0.0)
    return out


def frag_p(im, theta, beta):
    """Lognormal fragility: P(damage | IM)."""
    im = np.maximum(np.asarray(im, float), 1e-3)
    return norm.cdf((np.log(im) - np.log(theta)) / beta)


def fit_fragility(im, y, theta0=None, beta0=0.2):
    """Maximum-likelihood lognormal fragility from binary outcomes (Baker 2015). Returns theta, beta, -loglik."""
    im, y = np.asarray(im, float), np.asarray(y, bool)
    ok = np.isfinite(im) & (im > 0)
    im, y = im[ok], y[ok]
    if theta0 is None:
        theta0 = np.median(im[y]) if y.any() else np.percentile(im, 99)
    li = np.log(im)

    def nll(p):
        pr = np.clip(norm.cdf((li - p[0]) / np.exp(p[1])), 1e-12, 1 - 1e-12)
        return -(y * np.log(pr) + (~y) * np.log(1 - pr)).sum()
    res = minimize(nll, [np.log(theta0), np.log(beta0)], method='Nelder-Mead',
                   options={'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 5000})
    return float(np.exp(res.x[0])), float(np.exp(res.x[1])), float(res.fun)


def bootstrap_fragility(im, y, n=200, seed=1):
    rng = np.random.default_rng(seed)
    im, y = np.asarray(im, float), np.asarray(y, bool)
    th, be = [], []
    for _ in range(n):
        i = rng.integers(0, len(im), len(im))
        if y[i].sum() < 3:
            continue
        t, b, _ = fit_fragility(im[i], y[i])
        th.append(t); be.append(b)
    return np.array(th), np.array(be)


# ---------- project data loaders ----------
def load_towers(dirs):
    """Towers from Notebook 00, with exposure from Notebook 01 if available (else a distance-to-coast rule)."""
    t = pd.read_parquet(need(dirs['processed'] / 'towers.parquet', '00'))
    ex = dirs['processed'] / 'towers_exposure.parquet'
    if ex.exists():
        e = pd.read_parquet(ex)
        t = t.merge(e[['tower_id', 'z0', 'terrain_cat', 'exposure', 'exposure_src']], on='tower_id', how='left')
    else:
        t['z0'], t['terrain_cat'], t['exposure'], t['exposure_src'] = np.nan, np.nan, None, None
    miss = t.exposure.isna()
    t.loc[miss, 'exposure'] = np.where(t.loc[miss, 'dist_coast_km'] <= 2.0, 'off-sea', 'in-land')
    t.loc[miss, 'exposure_src'] = 'rule: ≤2 km from coast = off-sea, else in-land'
    t['terrain_cat'] = t.terrain_cat.fillna(2).astype(int)
    return t


def load_network(dirs):
    """Lines and substations (with population served) from Notebooks 00/01."""
    import geopandas as gpd
    gpkg = need(dirs['processed'] / 'odisha_network.gpkg', '00')
    lines = gpd.read_file(gpkg, layer='lines')
    subs = gpd.read_file(gpkg, layer='substations')
    popf = dirs['processed'] / 'substations_population.csv'
    if popf.exists():
        p = pd.read_csv(popf)
        subs = subs.merge(p[['sub_id', 'pop_worldpop']], on='sub_id', how='left')
        subs['pop'] = subs.pop_worldpop.fillna(subs.pop_weight)
        subs['pop_src'] = np.where(subs.pop_worldpop.notna(), 'WorldPop 2020 service area', 'Raj et al. weight')
    else:
        subs['pop'], subs['pop_src'] = subs.pop_weight, 'Raj et al. weight'
    subs['pop'] = subs['pop'].fillna(0.0)
    return lines, subs


def load_fragility(dirs, which='any_damage'):
    f = dirs['processed'] / 'fragility_params.json'
    if f.exists():
        d = json.loads(f.read_text())
        if which in d:
            return d[which]['theta'], d[which]['beta'], d[which].get('source', 'Notebook 02')
    print('  ! fragility_params.json not found: using Raj et al. collapse fragility (292.5 km/h, β = 0.104)')
    return 292.5 / 3.6, 0.104, 'Raj et al. (collapse)'


# ---------- risk engine (Notebooks 05 and 06) ----------
def line_matrix(line_pos, n_lines):
    """Sparse tower × line indicator used to combine tower failure probabilities into line failure probabilities."""
    n = len(line_pos)
    return csr_matrix((np.ones(n), (np.arange(n), line_pos)), shape=(n, n_lines))


def storm_probabilities(H, cell, gfac, theta_t, beta, M, scale=1.0, batch=512):
    """For every storm: expected damaged towers, line failure probabilities and per-tower annual sums.

    H: storms × cells storm-max 10-min wind; cell: grid cell of each tower; gfac: wind→IM factor per tower;
    theta_t: fragility median per tower (array); M: tower × line indicator."""
    n_s, n_l = H.shape[0], M.shape[1]
    E = np.zeros(n_s); P_line = np.zeros((n_s, n_l), np.float32); tower_sum = np.zeros(len(cell))
    lt = np.log(theta_t)
    for s in range(0, n_s, batch):
        im = H[s:s + batch][:, cell].astype(np.float32) * (gfac * scale)[None, :]
        p = norm.cdf((np.log(np.maximum(im, 1e-3)) - lt[None, :]) / beta)
        E[s:s + batch] = p.sum(1)
        tower_sum += p.sum(0)
        logq = (M.T @ np.log1p(-np.minimum(p, 1 - 1e-9)).T).T
        P_line[s:s + batch] = 1 - np.exp(logq)
    return E, P_line, tower_sum


def outage_mc(P_line, frm, to, n_sub, sub_pop, n_mc=30, min_p=1e-3, seed=7):
    """Expected population disconnected per storm (Raj et al.: population outside the largest connected component)."""
    rng = np.random.default_rng(seed)

    def lost(alive):
        g = csr_matrix((np.ones(alive.sum()), (frm[alive], to[alive])), shape=(n_sub, n_sub))
        nc, lab = connected_components(g, directed=False)
        big = np.bincount(lab).argmax()
        return sub_pop[lab != big].sum()
    base = lost(np.ones(len(frm), bool))
    out = np.zeros(P_line.shape[0])
    for s in np.where(P_line.max(1) > min_p)[0]:
        fails = rng.random((n_mc, P_line.shape[1])) < P_line[s][None, :]
        out[s] = np.mean([max(lost(~f) - base, 0.0) for f in fails])
    return out


def annual_from_events(values, sim_year, n_years):
    return np.bincount(np.asarray(sim_year, int), weights=np.asarray(values, float), minlength=n_years)


def rp_table(annual, rps=(10, 25, 50, 100, 250, 500)):
    return {T: float(np.quantile(annual, 1 - 1 / T)) for T in rps if T <= len(annual)}


def exceedance_curve(annual):
    a = np.sort(np.asarray(annual))[::-1]
    rank = np.arange(1, len(a) + 1)
    return pd.DataFrame({'annual_value': a, 'return_period_yr': len(a) / rank})
def build_risk_context(dirs, storm_10min_corr=1.0, kv_scope=None):
    """Load everything the risk engine needs: STORM hazard (03), towers with exposure (00/01), lines and population."""
    z = np.load(need(dirs['interim'] / 'hazard_storm_cells.npz', '03'), allow_pickle=False)
    th = pd.read_parquet(need(dirs['processed'] / 'towers_hazard.parquet', '03'))[['tower_id', 'cell']]
    towers = load_towers(dirs).merge(th, on='tower_id', how='inner')
    if kv_scope:
        towers = towers[towers.voltage_kv.isin(kv_scope)]
    lines, subs = load_network(dirs)
    lines = lines.sort_values('line_id').reset_index(drop=True)
    towers['line_pos'] = towers.line_id.map({lid: i for i, lid in enumerate(lines.line_id)})
    towers = towers.dropna(subset=['line_pos']).reset_index(drop=True)
    towers['line_pos'] = towers.line_pos.astype(int)
    n_sub = int(max(subs.sub_id.max(), lines.from_sub.max(), lines.to_sub.max()))
    sub_pop = np.zeros(n_sub)
    sub_pop[subs.sub_id.values.astype(int) - 1] = subs['pop'].values
    return dict(H=z['H'], years=z['sim_year'].astype(int), ids=z['storm_id'], n_years=int(z['n_years']),
                towers=towers, lines=lines, subs=subs, cell=towers.cell.values.astype(int),
                gfac=gust_factor(towers.exposure, 600) * storm_10min_corr, M=line_matrix(towers.line_pos.values, len(lines)),
                frm=lines.from_sub.values.astype(int) - 1, to=lines.to_sub.values.astype(int) - 1, n_sub=n_sub, sub_pop=sub_pop)


def risk_run(ctx, theta_t, beta, scale=1.0, n_mc=30, cost_cr=0.73, outage_days=7.0, kwh_pp_day=3.0, voll_inr=100.0, seed=7):
    """One full risk run over all STORM storms. Costs in INR crore. Returns events, annual totals and EAD."""
    theta_t = np.broadcast_to(np.asarray(theta_t, float), (len(ctx['cell']),))
    E, P_line, tsum = storm_probabilities(ctx['H'], ctx['cell'], ctx['gfac'], theta_t, beta, ctx['M'], scale)
    pop = outage_mc(P_line, ctx['frm'], ctx['to'], ctx['n_sub'], ctx['sub_pop'], n_mc=n_mc, seed=seed)
    ev = pd.DataFrame({'storm_id': ctx['ids'], 'sim_year': ctx['years'], 'exp_damaged_towers': E, 'exp_pop_disconnected': pop})
    ev['tower_cost_cr'] = E * cost_cr
    ev['ens_mwh'] = pop * outage_days * kwh_pp_day / 1000.0
    ev['outage_cost_cr'] = ev.ens_mwh * 1000.0 * voll_inr / 1e7
    ev['total_cost_cr'] = ev.tower_cost_cr + ev.outage_cost_cr
    cols = ['exp_damaged_towers', 'exp_pop_disconnected', 'tower_cost_cr', 'outage_cost_cr', 'total_cost_cr']
    annual = pd.DataFrame({c: annual_from_events(ev[c], ev.sim_year, ctx['n_years']) for c in cols})
    return dict(events=ev, annual=annual, P_line=P_line, tower_rate=tsum / ctx['n_years'],
                ead={c: float(annual[c].mean()) for c in cols})
# ==== end of shared functions ====

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

ROOT, DIRS = setup_project()

# ---------------- settings (keep the same values in Notebook 06) ----------------
FRAGILITY = 'any_damage'            # 'any_damage' or 'collapse' (Notebook 02 fits) or 'raj_collapse'
BETA_OVERRIDE = None                # e.g. 0.2 or 0.3: widen the fragility (β ≈ 0.10 from one storm is very steep)
CORROSION_THETA_FACTOR = 1.0        # e.g. 0.85 to test ~15% loss of capacity in old coastal lines (Li et al. 2022; review: ~20% in 25 yr)
COST_PER_DAMAGED_TOWER_CR = 93.5 / 128   # INR crore: Fani DLNA transmission damage (8,138.7 − 8,045.2 crore) ÷ 128 damaged towers
OUTAGE_DAYS = 7.0                   # days of supply lost at disconnected substations (transmission restoration)
KWH_PER_PERSON_DAY = 3.0            # energy served per person per day (all sectors); Odisha ≈ 1,600–1,900 kWh/person/yr incl. industry
VOLL_INR_PER_KWH = 100.0            # value of lost load (INR/kWh); test 40–250
N_MC = 30                           # Monte Carlo samples of line failures per storm
STORM_10MIN_CORR = 1.0              # same as Notebook 03
OBSERVED = {   # recorded transmission tower damage, 132/220 kV
    'FANI_2019': dict(observed=128, state='any_damage', source='OPTCL via RTI (Raj et al.): 87 collapsed + 41 partial'),
    'PHAILIN_2013': dict(observed=92, state='any_damage', source='Mohanty et al. 2020: 71 × 220 kV + 21 × 132 kV damaged'),
    'SUPERCYCLONE_1999': dict(observed=36, state='collapse', source='Mohanty et al. 2020: 36 towers collapsed (220/132 kV)'),
}

theta, beta, frag_src = load_fragility(DIRS, FRAGILITY)
if BETA_OVERRIDE:
    beta, frag_src = float(BETA_OVERRIDE), frag_src + f' with β overridden to {BETA_OVERRIDE}'
print(f'Fragility: {FRAGILITY} θ = {theta:.1f} m/s, β = {beta:.3f} ({frag_src})')
ctx = build_risk_context(DIRS, storm_10min_corr=STORM_10MIN_CORR)
print(f"Storms: {len(ctx['ids']):,} in {ctx['n_years']:,} years; towers in hazard domain: {len(ctx['cell']):,}; "
      f"lines: {len(ctx['lines'])}; population in model: {ctx['sub_pop'].sum() / 1e6:.1f} million ({ctx['subs'].pop_src.iloc[0]})")

## 1. Risk run

In [ ]:
theta_t = np.full(len(ctx['cell']), theta * CORROSION_THETA_FACTOR)
R = risk_run(ctx, theta_t, beta, n_mc=N_MC, cost_cr=COST_PER_DAMAGED_TOWER_CR, outage_days=OUTAGE_DAYS,
             kwh_pp_day=KWH_PER_PERSON_DAY, voll_inr=VOLL_INR_PER_KWH)
ev, annual, ead = R['events'], R['annual'], R['ead']
ev.to_parquet(DIRS['processed'] / 'risk_events.parquet', index=False)
annual.to_parquet(DIRS['processed'] / 'risk_annual.parquet', index=False)

rps = rp_table(annual.total_cost_cr.values, (10, 25, 50, 100, 250, 500))
rp_towers = rp_table(annual.exp_damaged_towers.values, (10, 25, 50, 100, 250, 500))
summary = dict(
    fragility=dict(state=FRAGILITY, theta=theta, beta=beta, source=frag_src, corrosion_factor=CORROSION_THETA_FACTOR),
    settings=dict(cost_per_damaged_tower_cr=COST_PER_DAMAGED_TOWER_CR, outage_days=OUTAGE_DAYS, kwh_per_person_day=KWH_PER_PERSON_DAY,
                  voll_inr_per_kwh=VOLL_INR_PER_KWH, n_mc=N_MC, storm_10min_corr=STORM_10MIN_CORR),
    EAD_cr=ead['total_cost_cr'], EAD_tower_cr=ead['tower_cost_cr'], EAD_outage_cr=ead['outage_cost_cr'],
    expected_damaged_towers_per_year=ead['exp_damaged_towers'], expected_person_outages_per_year=ead['exp_pop_disconnected'],
    loss_rp_cr={str(k): v for k, v in rps.items()}, damaged_towers_rp={str(k): v for k, v in rp_towers.items()},
    n_years=ctx['n_years'], n_storms=int(len(ev)))
(DIRS['processed'] / 'risk_summary.json').write_text(json.dumps(summary, indent=2))

print(f"Expected annual damage (EAD): INR {ead['total_cost_cr']:,.1f} crore/yr "
      f"(repairs {ead['tower_cost_cr']:,.1f}; outage {ead['outage_cost_cr']:,.1f})")
print(f"Expected towers damaged per year: {ead['exp_damaged_towers']:.1f}")
print('\nReturn-period losses (annual total, INR crore) and damaged towers:')
print(pd.DataFrame({'loss_cr': rps, 'damaged_towers': rp_towers}).round(1).to_string())

## 2. Which lines and towers carry the risk

In [ ]:
lines = ctx['lines'].copy()
lines['annual_failure_rate'] = R['P_line'].sum(0) / ctx['n_years']
tw = ctx['towers'][['tower_id', 'line_pos', 'lat', 'lon', 'voltage_kv', 'band', 'district']].copy()
tw['annual_damage_prob'] = R['tower_rate']
lines['exp_damaged_towers_yr'] = tw.groupby('line_pos').annual_damage_prob.sum().reindex(range(len(lines))).fillna(0).values
lines['towers_in_domain'] = tw.groupby('line_pos').size().reindex(range(len(lines))).fillna(0).astype(int).values

# corridor population weight (Raj et al. Unique_Edges) as the consequence of losing a line
try:
    ue = pd.read_excel(DIRS['external'] / 'raj_network' / 'Base case _ no strengthening_with FII index' / 'Network_details.xlsx',
                       sheet_name='Unique_Edges')
    key = lambda a, b: tuple(sorted((int(a), int(b))))
    pw = {key(a, b): p for a, b, p in zip(ue['Start Node'], ue['End Node'], ue['Population weight'])}
    lines['corridor_pop_weight'] = [pw.get(key(a, b), 0.0) for a, b in zip(lines.from_sub, lines.to_sub)]
except Exception as e:
    print('  ! corridor weights not found:', e); lines['corridor_pop_weight'] = 0.0
lines['risk_score'] = (lines.exp_damaged_towers_yr * COST_PER_DAMAGED_TOWER_CR +
                       lines.annual_failure_rate * lines.corridor_pop_weight * OUTAGE_DAYS * KWH_PER_PERSON_DAY * VOLL_INR_PER_KWH / 1e7)
lr = lines.drop(columns='geometry').sort_values('risk_score', ascending=False)
lr.to_csv(DIRS['tables'] / '05_line_risk.csv', index=False)
tw.to_parquet(DIRS['processed'] / 'tower_risk.parquet', index=False)
print(lr[['line_id', 'line_name', 'voltage_kv', 'towers_in_domain', 'annual_failure_rate', 'exp_damaged_towers_yr',
          'corridor_pop_weight', 'risk_score']].head(15).round(4).to_string(index=False))
print('\nExpected damaged towers per year by coastal band:')
print(tw.groupby('band').annual_damage_prob.sum().round(2).to_string())

## 3. Validation: historical cyclones through the same model
Same wind model and gust conversion as Notebook 02 (IMD 3-min winds × WMO G(3 s,180 s)). Today's network is used for all events, so older events are only indicative (the 1999 network was smaller).

In [ ]:
towers_all = load_towers(DIRS)
g180 = gust_factor(towers_all.exposure, 180)
frag_all = {k: load_fragility(DIRS, k)[:2] for k in ('any_damage', 'collapse')}
rows = []
for f in sorted(DIRS['processed'].glob('ibtracs_*.csv')):
    evn = f.stem.replace('ibtracs_', '')
    g = pd.read_csv(f, parse_dates=['ISO_TIME']).sort_values('ISO_TIME')
    w = g.NEWDELHI_WIND.fillna(g['WMO_WIND']) if 'WMO_WIND' in g else g.NEWDELHI_WIND
    m = w.notna() & g.LAT.notna()
    g, w = g[m], w[m]
    if len(g) < 2:
        continue
    t = (g.ISO_TIME - g.ISO_TIME.iloc[0]).dt.total_seconds().values / 3600
    rmw = (g.USA_RMW * NMI2KM).interpolate(limit_direction='both').fillna(30.0).values if 'USA_RMW' in g else np.full(len(g), 30.0)
    tr = interp_track(t, g.LAT.values, g.LON.values, w.values * KT2MS, rmw, g.get('NEWDELHI_PRES', pd.Series(np.nan, index=g.index)).values, dt_h=0.25)
    near = haversine_km(towers_all.lat.values[:, None], towers_all.lon.values[:, None], tr['lat'][None, ::8], tr['lon'][None, ::8]).min(1) < 400
    gust = np.zeros(len(towers_all))
    if near.any():
        gust[near] = holland_max_wind(tr, towers_all.lat.values[near], towers_all.lon.values[near]) * g180[near]
    scope = towers_all.voltage_kv.isin([132, 220]).values
    row = dict(event=evn, max_gust_at_towers=gust.max(), towers_gust_gt50=int((gust > 50).sum()))
    for k, (th_, be_) in frag_all.items():
        row[f'pred_{k}_132_220'] = frag_p(gust[scope], th_, be_).sum()
    if evn in OBSERVED:
        row.update(observed=OBSERVED[evn]['observed'], observed_state=OBSERVED[evn]['state'], observed_source=OBSERVED[evn]['source'])
    rows.append(row)
val = pd.DataFrame(rows)
if len(val):
    val.to_csv(DIRS['tables'] / '05_validation_events.csv', index=False)
    print(val.round(1).to_string(index=False))
    print('\nFani is the calibration event, so it should match by construction. Phailin and 1999 are the independent tests.')
else:
    print('No IBTrACS event files from Notebook 00; skipping validation.')

## 4. Figures

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(21, 6.5))
ec = exceedance_curve(annual.total_cost_cr.values)
ax[0].semilogx(ec.return_period_yr, ec.annual_value, lw=2)
for T in (25, 50, 100):
    ax[0].plot(T, rps[T], 'o', c='tab:red'); ax[0].annotate(f'{T}-yr: {rps[T]:,.0f} cr', (T, rps[T]), textcoords='offset points', xytext=(6, -12), fontsize=8)
ax[0].axhline(ead['total_cost_cr'], ls='--', c='grey'); ax[0].text(1.2, ead['total_cost_cr'], f"EAD {ead['total_cost_cr']:,.0f} cr/yr", va='bottom', fontsize=8)
ax[0].set_xlim(1, ctx['n_years']); ax[0].set_xlabel('return period (years)'); ax[0].set_ylabel('annual loss (INR crore)')
ax[0].set_title('Loss-exceedance curve (repairs + outage)'); ax[0].grid(alpha=0.3)

ax[1].bar(['repairs', 'outage'], [ead['tower_cost_cr'], ead['outage_cost_cr']], color=['tab:orange', 'tab:blue'])
ax[1].set_ylabel('EAD (INR crore/yr)'); ax[1].set_title('What drives expected annual damage')

sc = ax[2].scatter(tw.lon, tw.lat, c=np.log10(np.maximum(tw.annual_damage_prob, 1e-6)), s=2, cmap='magma_r', vmin=-5, vmax=-1)
cb = plt.colorbar(sc, ax=ax[2], shrink=0.7); cb.set_label('annual damage probability (log10)')
ax[2].set_aspect('equal'); ax[2].set_title('Annual probability of damage per tower')
plt.tight_layout(); fig.savefig(DIRS['figures'] / '05_risk.png', dpi=200); plt.show()